In [ ]:
import shutil
import random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

INPUT_ROOT = Path('/kaggle/input')
TRAIN_DIR = Path('/kaggle/working/rec_train')
VAL_DIR = Path('/kaggle/working/rec_val')
STRANGER_DIR = Path('/kaggle/working/rec_strangers')

for d in [TRAIN_DIR, VAL_DIR, STRANGER_DIR]:
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

# 1. Tự động tìm đúng đường dẫn của 3 bộ dữ liệu
esp_person1 = next((d for d in INPUT_ROOT.rglob('train/person-001') if d.is_dir()), None)
assert esp_person1 is not None, "Chưa gắn bộ esp32-face-model2 vào Input!"
esp_root = esp_person1.parent.parent

sample_pins = next((d for d in INPUT_ROOT.rglob('pins_*') if d.is_dir()), None)
assert sample_pins is not None, "Chưa gắn bộ Pins Face Recognition vào Input!"
pins_root = sample_pins.parent

lfw_imgs = [p for p in INPUT_ROOT.rglob('*.jpg') if 'lfw' in str(p).lower()]
assert len(lfw_imgs) > 0, "Chưa gắn bộ LFW Dataset vào Input!"

print("[1/3] Đã nhận diện đủ 3 bộ dữ liệu. Đang gom danh sách ảnh...")
copy_tasks = []

# 2. Lấy 105 người từ Pins Face Recognition
for person_dir in pins_root.glob('pins_*'):
    if person_dir.is_dir():
        dst = TRAIN_DIR / person_dir.name
        dst.mkdir(parents=True, exist_ok=True)
        for img in person_dir.glob('*.*'):
            copy_tasks.append((img, dst / img.name))

# 3. Lấy 4 người nhóm bạn (nhân bản 4x tập train để cân bằng số lượng với Pins Face)
for p_dir in sorted((esp_root / 'train').glob('person-*')):
    dst = TRAIN_DIR / p_dir.name
    dst.mkdir(parents=True, exist_ok=True)
    for img in p_dir.glob('*.*'):
        for i in range(4):
            copy_tasks.append((img, dst / f"c{i}_{img.name}"))

for p_dir in sorted((esp_root / 'val').glob('person-*')):
    dst = VAL_DIR / p_dir.name
    dst.mkdir(parents=True, exist_ok=True)
    for img in p_dir.glob('*.*'):
        copy_tasks.append((img, dst / img.name))

# 4. Lấy 200 ảnh người lạ từ LFW để kiểm tra ngưỡng từ chối mở cửa
random.seed(42)
random.shuffle(lfw_imgs)
for idx, img in enumerate(lfw_imgs[:200]):
    copy_tasks.append((img, STRANGER_DIR / f"stranger_{idx:03d}.jpg"))

print(f"[2/3] Đang copy song song {len(copy_tasks)} ảnh...")
with ThreadPoolExecutor(max_workers=16) as ex:
    ex.map(lambda pair: shutil.copyfile(pair[0], pair[1]), copy_tasks)

print("[3/3] HOÀN TẤT!")
print(f"-> Tổng số người huấn luyện: {len(list(TRAIN_DIR.iterdir()))} người")
print(f"-> Ảnh Train: {len(list(TRAIN_DIR.rglob('*.*')))} | Ảnh Val ESP32: {len(list(VAL_DIR.rglob('*.*')))} | Ảnh Người lạ: {len(list(STRANGER_DIR.rglob('*.*')))}")

In [ ]:
!pip install -q timm scikit-learn
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm

train_tf = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.25),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

train_ds = datasets.ImageFolder('/kaggle/working/rec_train', transform=train_tf)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
num_classes = len(train_ds.classes)

class ArcFaceHead(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.50):
        super().__init__()
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.s, self.m = s, m

    def forward(self, embeddings, labels):
        cosine = F.linear(F.normalize(embeddings), F.normalize(self.weight))
        theta = torch.acos(torch.clamp(cosine, -1.0 + 1e-6, 1.0 - 1e-6))
        target_logits = torch.cos(theta + self.m)
        one_hot = F.one_hot(labels, num_classes=self.weight.size(0)).float()
        return (one_hot * target_logits + (1.0 - one_hot) * cosine) * self.s

class FaceEmbedder(nn.Module):
    def __init__(self, emb_size=512):
        super().__init__()
        self.backbone = timm.create_model('mobilenetv3_large_100', pretrained=True, num_classes=0)
        with torch.no_grad():
            in_features = self.backbone(torch.randn(1, 3, 112, 112)).shape[1]
        self.neck = nn.Sequential(
            nn.Linear(in_features, emb_size, bias=False),
            nn.BatchNorm1d(emb_size)
        )

    def forward(self, x):
        return F.normalize(self.neck(self.backbone(x)), p=2, dim=1)

device = torch.device('cuda')
embedder = FaceEmbedder(emb_size=512).to(device)
arcface = ArcFaceHead(in_features=512, out_features=num_classes).to(device)

optimizer = torch.optim.AdamW(list(embedder.parameters()) + list(arcface.parameters()), lr=4e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda')

for epoch in range(1, 21):
    embedder.train()
    arcface.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            emb = embedder(imgs)
            logits = arcface(emb, labels)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * imgs.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    scheduler.step()
    print(f"Epoch {epoch:02d}/20 | Loss: {total_loss/total:.4f} | Acc: {correct/total*100:.2f}%")

torch.save(embedder.state_dict(), '/kaggle/working/rec_embedder.pth')
print("Đã lưu trọng số rec_embedder.pth!")

In [ ]:
!pip install -q onnx onnxscript
import torch
import numpy as np
from pathlib import Path
from PIL import Image
from torchvision import transforms
from sklearn.cluster import KMeans

embedder.eval()

# 1. Xuất model trích xuất vector 512 chiều sang ONNX
dummy = torch.randn(1, 3, 112, 112, device=device)
torch.onnx.export(
    embedder, dummy, '/kaggle/working/rec_arcface.onnx',
    input_names=['input'], output_names=['embedding'],
    dynamic_axes={'input': {0: 'batch'}, 'embedding': {0: 'batch'}},
    opset_version=12,
    dynamo=False
)

val_tf = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

stranger_tf = transforms.Compose([
    transforms.CenterCrop(150),
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

@torch.no_grad()
def get_emb(img_path, is_stranger=False):
    img = Image.open(img_path).convert('RGB')
    tf = stranger_tf if is_stranger else val_tf
    t = tf(img).unsqueeze(0).to(device)
    return embedder(t).cpu().numpy()[0]

# 2. Tạo database vector đa góc độ (4 vector đại diện/người) cho 4 thành viên nhóm
face_db = {}
for p_dir in sorted(Path('/kaggle/working/rec_train').glob('person-*')):
    orig_imgs = [p for p in p_dir.glob('c0_*.*')]
    embs = np.array([get_emb(p) for p in orig_imgs])
    kmeans = KMeans(n_clusters=min(4, len(embs)), random_state=42, n_init=10).fit(embs)
    centers = kmeans.cluster_centers_
    centers = centers / np.linalg.norm(centers, axis=1, keepdims=True)
    face_db[p_dir.name] = centers

np.save('/kaggle/working/face_db.npy', face_db)

# 3. Kiểm tra độ chính xác và khoảng cách điểm Người nhà vs Người lạ
def match_face(emb):
    best_name, best_score = "Unknown", -1.0
    for name, centers in face_db.items():
        score = float(np.max(centers @ emb))
        if score > best_score:
            best_score, best_name = score, name
    return best_name, best_score

same_scores, correct_id, total_val = [], 0, 0
for p_dir in sorted(Path('/kaggle/working/rec_val').glob('person-*')):
    for img in p_dir.glob('*.*'):
        pred_name, score = match_face(get_emb(img))
        same_scores.append(score)
        if pred_name == p_dir.name:
            correct_id += 1
        total_val += 1

stranger_scores = [match_face(get_emb(img, is_stranger=True))[1] for img in Path('/kaggle/working/rec_strangers').glob('*.*')]

print(f"\nĐộ chính xác định danh 4 người (Val): {correct_id}/{total_val} ({correct_id/total_val*100:.2f}%)")
print(f"Điểm tương đồng trung bình NGƯỜI NHÀ: {np.mean(same_scores):.4f} (Thấp nhất: {np.min(same_scores):.4f})")
print(f"Điểm tương đồng trung bình NGƯỜI LẠ:  {np.mean(stranger_scores):.4f} (Cao nhất: {np.max(stranger_scores):.4f})")
print(f"-> Ngưỡng mở khóa đề xuất (Threshold): {(np.mean(same_scores) + np.mean(stranger_scores))/2:.2f}")